# 🤖 Detecção de Anomalias com Isolation Forest

In [1]:
import sqlite3
import pandas as pd
from sklearn.ensemble import IsolationForest

# Conectar ao banco de dados e carregar os logs
con = sqlite3.connect("../database.db")
logs = pd.read_sql_query("SELECT * FROM logs", con)

# Converter data_hora para datetime
logs['data_hora'] = pd.to_datetime(logs['data_hora'])

# Criar colunas simuladas: duração da sessão e páginas visitadas
logs['session_duration'] = logs.groupby("usuario")["data_hora"].transform(lambda x: (x.max() - x.min()).total_seconds())
logs['pages_visited'] = logs.groupby("usuario")["acao"].transform("count")

# Substituir valores ausentes por padrão
logs['session_duration'] = logs['session_duration'].fillna(0)
logs['pages_visited'] = logs['pages_visited'].fillna(1)

# Normalizar as colunas (min-max)
logs['session_duration'] = (logs['session_duration'] - logs['session_duration'].min()) / (logs['session_duration'].max() - logs['session_duration'].min())
logs['pages_visited'] = (logs['pages_visited'] - logs['pages_visited'].min()) / (logs['pages_visited'].max() - logs['pages_visited'].min())

# Aplicar Isolation Forest
model = IsolationForest(contamination=0.2, random_state=42)
logs['anomaly_score'] = model.fit_predict(logs[['session_duration', 'pages_visited']])

# Classificar como 'Normal' ou 'Suspeito'
logs['classification'] = logs['anomaly_score'].apply(lambda x: 'Normal' if x == 1 else 'Suspeito')

# Exibir resultado
logs.head()


,id,usuario,acao,data_hora,session_duration,pages_visited,anomaly_score,classification
0,1,Teodoro,Comprou ingresso para: Show A,2025-05-19 16:24:46,1.0,1.0,1,Normal
1,2,Teodoro,Comprou ingresso para: Show A,2025-05-19 16:24:48,1.0,1.0,1,Normal
2,3,Teodoro,Comprou ingresso para: Show A,2025-05-19 16:24:49,1.0,1.0,1,Normal
3,4,Teodoro,Comprou ingresso para: Show B,2025-05-19 16:24:52,1.0,1.0,1,Normal
4,5,Teodoro,Comprou ingresso para: Show B,2025-05-19 16:24:52,1.0,1.0,1,Normal
